This visualization notebook is initially taken from Tanya Lomskaya on Plotly Dash. GitHub code of her can be found here:
https://github.com/lomska/Visualizing-Global-Trade-Networks/blob/main/Building_a_Network_Graph_of_Global_Trade.ipynb

In [ ]:
import dash
import dash_cytoscape as cyto
import pandas as pd
from dash import html, dcc, Output, Input, dash_table
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

# Load extra layouts
cyto.load_extra_layouts()

# Load data
df_notes = pd.read_csv("../data/df_noteId_Soft_top3_clusters.csv")
df_topic_names = pd.read_csv("../data/Topic_Names.csv")  # Load topic names

# Create a dictionary mapping topic numbers to names
topic_name_map = dict(zip(df_topic_names["TopicNumber"], df_topic_names["TopicName"]))

# Shorten long topic names for better node display
def shorten_name(name, max_length=15):
    return name if len(name) <= max_length else name[:max_length] + "..."

# Create the NetworkX graph
def create_bertopic_network(df_notes):
    G = nx.Graph()
    node_sizes = {}
    topic_to_notes = {}
    
    for _, row in df_notes.iterrows():
        for topic in [row["BERTopic_number1"]]:  # Only considering BERTopic_number1
            node_sizes[topic] = node_sizes.get(topic, 0) + 1
            if topic not in topic_to_notes:
                topic_to_notes[topic] = []
            topic_to_notes[topic].append(row["noteId"])
    
    for node, size in node_sizes.items():
        G.add_node(node, size=size)
    
    edge_weights = {}
    for _, row in df_notes.iterrows():
        topics = {row["BERTopic_number1"], row["BERTopic_number2"], row["BERTopic_number3"]}
        for topic1 in topics:
            for topic2 in topics:
                if topic1 != topic2:
                    edge = tuple(sorted((topic1, topic2)))
                    edge_weights[edge] = edge_weights.get(edge, 0) + 1
    
    for (node1, node2), weight in edge_weights.items():
        G.add_edge(node1, node2, weight=weight)
    
    return G, node_sizes, topic_to_notes

G, node_sizes, topic_to_notes = create_bertopic_network(df_notes)

# Generate a discrete color mapping for topics
unique_topics = list(node_sizes.keys())  
color_map = plt.get_cmap("Set3")  # Discrete colormap for distinct colors
topic_colors = {topic: color_map(i / len(unique_topics)) for i, topic in enumerate(unique_topics)}

# Convert to CSS rgba format
def rgba_to_css(rgba_tuple):
    return f"rgba({int(rgba_tuple[0]*255)}, {int(rgba_tuple[1]*255)}, {int(rgba_tuple[2]*255)}, 1)"

# Logarithmic scaling for better node size contrast
def scale_node_size(value, min_value, max_value, default_value=10, min_size=15, max_size=150):
    """ 
    Scales node sizes **logarithmically** for better contrast.
    """
    if max_value == min_value:  # Edge case: all nodes have same size
        return min_size  

    # Apply **logarithmic scaling** to avoid flattening differences
    log_min = np.log1p(min_value)  # log(1 + min_value)
    log_max = np.log1p(max_value)  # log(1 + max_value)
    log_value = np.log1p(value)    # log(1 + value)

    # Normalize the logarithmic value
    norm_value = (log_value - log_min) / (log_max - log_min)

    # Scale it to the final node size range
    scaled_size = min_size + (max_size - min_size) * norm_value

    # Ensure **default nodes (10 occurrences) remain smaller than real small nodes**
    if value == default_value:
        return min_size - 3  
    
    return scaled_size

# Convert NetworkX graph to Cytoscape elements with colors
def generate_cytoscape_elements(G, node_sizes):
    elements = []
    
    min_occurrences = min(node_sizes.values())
    max_occurrences = max(node_sizes.values())
    max_weight = max((edge[2]["weight"] for edge in G.edges(data=True)), default=1)

    for node in G.nodes():
        full_name = topic_name_map.get(node, str(node))  
        node_label = shorten_name(full_name)  
        node_color = rgba_to_css(topic_colors.get(node, (0.5, 0.5, 0.5)))  

        # Apply corrected scaling function call
        node_size = scale_node_size(
            node_sizes.get(node, 10),  
            min_value=min_occurrences,  
            max_value=max_occurrences,  
            default_value=10,  
            min_size=15,  
            max_size=150  
        )

        elements.append({
            "data": {
                "id": str(node),
                "label": node_label,
                "full_label": full_name,
                "size": node_size,
                "background_color": node_color
            }
        })
    
    for edge in G.edges(data=True):
        weight = edge[2].get("weight", 1)

        # Scale edge weights to a reasonable range
        scaled_weight = 1 + (5 * (weight / max_weight))

        elements.append({
            "data": {
                "source": str(edge[0]),
                "target": str(edge[1]),
                "weight": scaled_weight
            }
        })
    
    return elements    

elements = generate_cytoscape_elements(G, node_sizes)

# Define Cytoscape styles
stylesheet = [
    {
        "selector": "node",
        "style": {
            "label": "data(label)",
            "background-color": "data(background_color)",
            "width": "data(size)",
            "height": "data(size)",
            "text-valign": "center",
            "text-halign": "center",
            "font-size": "mapData(size, 15, 150, 10, 20)",  
            "font-family": "Arial, sans-serif",
            "color": "black",
        }
    },
    {
        "selector": "edge",
        "style": {
            "width": "data(weight)",
            "line-color": "#888",
            "curve-style": "bezier",
        }
    }
]

# Build Dash App
app = dash.Dash(__name__)
app.layout = html.Div(style={"display": "flex", "height": "90vh", "font-family": "Arial, sans-serif"}, children=[
    html.Div([
        cyto.Cytoscape(
            id="cytoscape",
            elements=elements,
            layout={
                "name": "fcose", 
                "nodeSeparation": 250,  
                "nodeRepulsion": 12000,  
                "idealEdgeLength": 200,  
                "edgeElasticity": 0.05,  
                "numIter": 2500  
            },
            style={"width": "75vw", "height": "100%", "background-color": "white", "position": "relative"},
            stylesheet=stylesheet
        )
    ], style={"flex": "3", "position": "relative"}),

    # **✅ Sidebar restored**
    html.Div([
        html.H3("Selected Topic Details", style={"font-family": "Arial, sans-serif"}),
        html.Div(id="topic-details", style={"margin-bottom": "10px"}),
        dash_table.DataTable(
            id="topic-table", 
            columns=[{"name": "Note ID", "id": "noteId"}], 
            page_size=10,
            style_table={"font-family": "Arial, sans-serif"}
        )
    ], style={"flex": "1", "padding": "10px", "background": "#f9f9f9", "overflow": "auto"})
])

# Callback to display full topic name and notes
@app.callback(
    [Output("topic-details", "children"), Output("topic-table", "data")],
    [Input("cytoscape", "tapNode")]
)
def display_topic_details(node_data):
    if node_data is None:
        return "Click a topic node to see details. This might take a few seconds.", []
    
    topic_id = int(node_data["data"]["id"])
    full_label = node_data["data"].get("full_label", str(topic_id))
    notes = topic_to_notes.get(topic_id, [])
    return f"Topic: {full_label}", [{"noteId": note} for note in notes]

if __name__ == "__main__":
    app.run_server(debug=True)
